In [0]:
# Create Silver schema

spark.sql("""
CREATE SCHEMA IF NOT EXISTS automotive_warranty.silver
""")

print("Silver schema created successfully!")

Silver schema created successfully!


In [0]:
# ============================================================
# SILVER LAYER - SOURCE TABLE INSPECTION
# ============================================================

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("SILVER SOURCE TABLE INSPECTION")
print("=" * 80)

for table in tables:
    df = spark.table(f"automotive_warranty.bronze.{table}")

    print(f"\n{table}")
    print("-" * 50)
    print("Columns:", df.columns)
    print("Rows:", df.count())

print("\n" + "=" * 80)
print("SOURCE INSPECTION COMPLETED")
print("=" * 80)

SILVER SOURCE TABLE INSPECTION

addresses
--------------------------------------------------
Columns: ['address_id', 'street_address', 'city', 'state', 'postal_code', 'country', 'ingestion_timestamp']
Rows: 10100

customer_vehicles
--------------------------------------------------
Columns: ['ownership_id', 'customer_id', 'vehicle_id', 'registration_plate', 'purchase_dealer_id', 'purchase_date', 'is_active', 'ingestion_timestamp']
Rows: 10000

customers
--------------------------------------------------
Columns: ['customer_id', 'first_name', 'last_name', 'email', 'phone_number', 'address_id', 'created_at', 'ingestion_timestamp']
Rows: 10000

dealers
--------------------------------------------------
Columns: ['dealer_id', 'dealer_name', 'license_number', 'address_id', 'contact_phone', 'ingestion_timestamp']
Rows: 20

parts_catalog
--------------------------------------------------
Columns: ['part_id', 'part_number', 'part_name', 'unit_price', 'category', 'ingestion_timestamp']
Rows: 50

In [0]:
# ============================================================
# SILVER LAYER - COMMON DATA CLEANING
# ============================================================

from pyspark.sql.functions import col, trim, when

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("SILVER LAYER - COMMON CLEANING")
print("=" * 80)

for table in tables:

    print(f"\nProcessing: {table}")

    # Read Bronze table
    df = spark.table(f"automotive_warranty.bronze.{table}")

    # --------------------------------------------------------
    # 1. Trim whitespace from string columns
    # 2. Convert empty strings to NULL
    # --------------------------------------------------------
    for field in df.schema.fields:

        if field.dataType.simpleString() == "string":

            df = df.withColumn(
                field.name,
                when(
                    trim(col(field.name)) == "",
                    None
                ).otherwise(
                    trim(col(field.name))
                )
            )

    # --------------------------------------------------------
    # 3. Remove completely duplicate records
    # --------------------------------------------------------
    df = df.dropDuplicates()

    # --------------------------------------------------------
    # 4. Write cleaned data to Silver as Delta
    # --------------------------------------------------------
    df.write \
      .format("delta") \
      .mode("overwrite") \
      .saveAsTable(
          f"automotive_warranty.silver.{table}"
      )

    print(f"Created Silver table: automotive_warranty.silver.{table}")

print("\n" + "=" * 80)
print("SILVER COMMON CLEANING COMPLETED")
print("=" * 80)

SILVER LAYER - COMMON CLEANING

Processing: addresses
Created Silver table: automotive_warranty.silver.addresses

Processing: customer_vehicles
Created Silver table: automotive_warranty.silver.customer_vehicles

Processing: customers
Created Silver table: automotive_warranty.silver.customers

Processing: dealers
Created Silver table: automotive_warranty.silver.dealers

Processing: parts_catalog
Created Silver table: automotive_warranty.silver.parts_catalog

Processing: parts_used
Created Silver table: automotive_warranty.silver.parts_used

Processing: service_appointments
Created Silver table: automotive_warranty.silver.service_appointments

Processing: service_centers
Created Silver table: automotive_warranty.silver.service_centers

Processing: service_order_tasks
Created Silver table: automotive_warranty.silver.service_order_tasks

Processing: service_orders
Created Silver table: automotive_warranty.silver.service_orders

Processing: service_types
Created Silver table: automotive_war

In [0]:
# ============================================================
# SILVER LAYER - VALIDATION
# ============================================================

tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("=" * 80)
print("SILVER TABLE VALIDATION")
print("=" * 80)

for table in tables:

    df = spark.table(f"automotive_warranty.silver.{table}")

    total_rows = df.count()
    duplicate_rows = total_rows - df.distinct().count()

    # Count NULL values
    null_row = df.select([
        __import__("pyspark").sql.functions.count(
            __import__("pyspark").sql.functions.when(
                __import__("pyspark").sql.functions.col(c).isNull(), c
            )
        ).alias(c)
        for c in df.columns
    ]).first()

    total_nulls = sum(null_row)

    print(f"\nTable: {table}")
    print(f"  Rows              : {total_rows:,}")
    print(f"  Duplicate rows    : {duplicate_rows:,}")
    print(f"  NULL values       : {total_nulls:,}")

print("\n" + "=" * 80)
print("SILVER VALIDATION COMPLETED")
print("=" * 80)

SILVER TABLE VALIDATION

Table: addresses
  Rows              : 10,100
  Duplicate rows    : 0
  NULL values       : 0

Table: customer_vehicles
  Rows              : 10,000
  Duplicate rows    : 0
  NULL values       : 0

Table: customers
  Rows              : 10,000
  Duplicate rows    : 0
  NULL values       : 0

Table: dealers
  Rows              : 20
  Duplicate rows    : 0
  NULL values       : 0

Table: parts_catalog
  Rows              : 50
  Duplicate rows    : 0
  NULL values       : 0

Table: parts_used
  Rows              : 500,000
  Duplicate rows    : 0
  NULL values       : 0

Table: service_appointments
  Rows              : 10,000
  Duplicate rows    : 0
  NULL values       : 0

Table: service_centers
  Rows              : 50
  Duplicate rows    : 0
  NULL values       : 0

Table: service_order_tasks
  Rows              : 500,000
  Duplicate rows    : 0
  NULL values       : 0

Table: service_orders
  Rows              : 10,000
  Duplicate rows    : 0
  NULL values    

In [0]:
# ============================================================
# SILVER LAYER - CUSTOMERS TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, lower, when

# Read Bronze customers
df_customers = spark.table(
    "automotive_warranty.bronze.customers"
)

# Clean string columns
string_columns = [
    field.name
    for field in df_customers.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_customers = df_customers.withColumn(
        c,
        trim(col(c))
    )

# Standardize email addresses if the column exists
if "email" in df_customers.columns:
    df_customers = df_customers.withColumn(
        "email",
        lower(trim(col("email")))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_customers = df_customers.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate customer records
df_customers = df_customers.dropDuplicates()

# Write Silver table
df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.customers"
    )

print("Silver customers transformation completed!")
print(f"Rows: {df_customers.count():,}")

Silver customers transformation completed!
Rows: 10,000


In [0]:
# ============================================================
# SILVER LAYER - VEHICLES TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, upper, when

# Read Bronze vehicles
df_vehicles = spark.table(
    "automotive_warranty.bronze.vehicles"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_vehicles.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_vehicles = df_vehicles.withColumn(
        c,
        trim(col(c))
    )

# Standardize VIN if available
if "vin" in df_vehicles.columns:
    df_vehicles = df_vehicles.withColumn(
        "vin",
        upper(trim(col("vin")))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_vehicles = df_vehicles.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate vehicle records
df_vehicles = df_vehicles.dropDuplicates()

# Write Silver vehicles table
df_vehicles.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.vehicles"
    )

print("Silver vehicles transformation completed!")
print(f"Rows: {df_vehicles.count():,}")

Silver vehicles transformation completed!
Rows: 10,000


In [0]:
# ============================================================
# SILVER LAYER - VEHICLES TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, upper, when

# Read Bronze vehicles
df_vehicles = spark.table(
    "automotive_warranty.bronze.vehicles"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_vehicles.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_vehicles = df_vehicles.withColumn(
        c,
        trim(col(c))
    )

# Standardize VIN if available
if "vin" in df_vehicles.columns:
    df_vehicles = df_vehicles.withColumn(
        "vin",
        upper(trim(col("vin")))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_vehicles = df_vehicles.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate vehicle records
df_vehicles = df_vehicles.dropDuplicates()

# Write Silver vehicles table
df_vehicles.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.vehicles"
    )

print("Silver vehicles transformation completed!")
print(f"Rows: {df_vehicles.count():,}")

Silver vehicles transformation completed!
Rows: 10,000


In [0]:
# ============================================================
# SILVER LAYER - VEHICLE MODELS TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, when

# Read Bronze vehicle_models
df_vehicle_models = spark.table(
    "automotive_warranty.bronze.vehicle_models"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_vehicle_models.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_vehicle_models = df_vehicle_models.withColumn(
        c,
        trim(col(c))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_vehicle_models = df_vehicle_models.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate records
df_vehicle_models = df_vehicle_models.dropDuplicates()

# Write Silver table
df_vehicle_models.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.vehicle_models"
    )

print("Silver vehicle_models transformation completed!")
print(f"Rows: {df_vehicle_models.count():,}")

Silver vehicle_models transformation completed!
Rows: 10


In [0]:
# ============================================================
# SILVER LAYER - CUSTOMER VEHICLES TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, when

# Read Bronze customer_vehicles
df_customer_vehicles = spark.table(
    "automotive_warranty.bronze.customer_vehicles"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_customer_vehicles.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_customer_vehicles = df_customer_vehicles.withColumn(
        c,
        trim(col(c))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_customer_vehicles = df_customer_vehicles.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate records
df_customer_vehicles = df_customer_vehicles.dropDuplicates()

# Write Silver table
df_customer_vehicles.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.customer_vehicles"
    )

print("Silver customer_vehicles transformation completed!")
print(f"Rows: {df_customer_vehicles.count():,}")

Silver customer_vehicles transformation completed!
Rows: 10,000


In [0]:
# ============================================================
# SILVER LAYER - SERVICE CENTERS TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, when

# Read Bronze service_centers
df_service_centers = spark.table(
    "automotive_warranty.bronze.service_centers"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_service_centers.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_service_centers = df_service_centers.withColumn(
        c,
        trim(col(c))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_service_centers = df_service_centers.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate service center records
df_service_centers = df_service_centers.dropDuplicates()

# Write Silver table
df_service_centers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.service_centers"
    )

print("Silver service_centers transformation completed!")
print(f"Rows: {df_service_centers.count():,}")

Silver service_centers transformation completed!
Rows: 50


In [0]:
# ============================================================
# SILVER LAYER - TECHNICIANS TRANSFORMATION
# ============================================================

from pyspark.sql.functions import col, trim, when

# Read Bronze technicians
df_technicians = spark.table(
    "automotive_warranty.bronze.technicians"
)

# Clean all string columns
string_columns = [
    field.name
    for field in df_technicians.schema.fields
    if field.dataType.simpleString() == "string"
]

for c in string_columns:
    df_technicians = df_technicians.withColumn(
        c,
        trim(col(c))
    )

# Convert blank strings to NULL
for c in string_columns:
    df_technicians = df_technicians.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# Remove duplicate technician records
df_technicians = df_technicians.dropDuplicates()

# Write Silver table
df_technicians.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "automotive_warranty.silver.technicians"
    )

print("Silver technicians transformation completed!")
print(f"Rows: {df_technicians.count():,}")

Silver technicians transformation completed!
Rows: 200


In [0]:
# ============================================================
# SILVER LAYER - REMAINING TABLES
# ============================================================

from pyspark.sql.functions import col, trim, when

remaining_tables = [
    "service_types",
    "service_appointments",
    "service_orders",
    "service_order_tasks",
    "parts_catalog",
    "parts_used",
    "dealers",
    "addresses",
    "warranty_claims"
]

for table_name in remaining_tables:

    print(f"Processing: {table_name}")

    # Read Bronze table
    df = spark.table(
        f"automotive_warranty.bronze.{table_name}"
    )

    # Find string columns
    string_columns = [
        field.name
        for field in df.schema.fields
        if field.dataType.simpleString() == "string"
    ]

    # Trim strings and convert blanks to NULL
    for c in string_columns:
        df = df.withColumn(
            c,
            trim(col(c))
        )

        df = df.withColumn(
            c,
            when(col(c) == "", None).otherwise(col(c))
        )

    # Remove duplicate records
    df = df.dropDuplicates()

    # Write Silver table
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(
            f"automotive_warranty.silver.{table_name}"
        )

    print(f"Completed: {table_name} | Rows: {df.count():,}")

print("==========================================")
print("All remaining Silver tables completed!")
print("==========================================")

Processing: service_types
Completed: service_types | Rows: 6
Processing: service_appointments
Completed: service_appointments | Rows: 10,000
Processing: service_orders
Completed: service_orders | Rows: 10,000
Processing: service_order_tasks
Completed: service_order_tasks | Rows: 500,000
Processing: parts_catalog
Completed: parts_catalog | Rows: 50
Processing: parts_used
Completed: parts_used | Rows: 500,000
Processing: dealers
Completed: dealers | Rows: 20
Processing: addresses
Completed: addresses | Rows: 10,100
Processing: warranty_claims
Completed: warranty_claims | Rows: 3,500
All remaining Silver tables completed!


In [0]:
# ============================================================
# SILVER LAYER - FINAL DATA QUALITY VALIDATION
# ============================================================

from pyspark.sql.functions import col, count, when

silver_tables = [
    "addresses",
    "customer_vehicles",
    "customers",
    "dealers",
    "parts_catalog",
    "parts_used",
    "service_appointments",
    "service_centers",
    "service_order_tasks",
    "service_orders",
    "service_types",
    "technicians",
    "vehicle_models",
    "vehicles",
    "warranty_claims"
]

print("============================================================")
print("SILVER LAYER DATA QUALITY SUMMARY")
print("============================================================")

for table_name in silver_tables:

    df = spark.table(
        f"automotive_warranty.silver.{table_name}"
    )

    # Row count
    row_count = df.count()

    # Duplicate count
    duplicate_count = row_count - df.dropDuplicates().count()

    # NULL count
    null_count = 0

    for c in df.columns:
        null_count += df.filter(
            col(c).isNull()
        ).count()

    print(
        f"{table_name:25} | "
        f"Rows: {row_count:8,} | "
        f"Duplicates: {duplicate_count:5,} | "
        f"NULLs: {null_count:,}"
    )

print("============================================================")
print("Silver Layer Validation Completed!")
print("============================================================")

SILVER LAYER DATA QUALITY SUMMARY
addresses                 | Rows:   10,100 | Duplicates:     0 | NULLs: 0
customer_vehicles         | Rows:   10,000 | Duplicates:     0 | NULLs: 0
customers                 | Rows:   10,000 | Duplicates:     0 | NULLs: 0
dealers                   | Rows:       20 | Duplicates:     0 | NULLs: 0
parts_catalog             | Rows:       50 | Duplicates:     0 | NULLs: 0
parts_used                | Rows:  500,000 | Duplicates:     0 | NULLs: 0
service_appointments      | Rows:   10,000 | Duplicates:     0 | NULLs: 0
service_centers           | Rows:       50 | Duplicates:     0 | NULLs: 0
service_order_tasks       | Rows:  500,000 | Duplicates:     0 | NULLs: 0
service_orders            | Rows:   10,000 | Duplicates:     0 | NULLs: 0
service_types             | Rows:        6 | Duplicates:     0 | NULLs: 0
technicians               | Rows:      200 | Duplicates:     0 | NULLs: 0
vehicle_models            | Rows:       10 | Duplicates:     0 | NULLs: 0
vehi